# KCC Language Detection — Row-Level + District Summary

## What this notebook does

For every district CSV it adds **two new columns**:
- `kccans_language` — language of the `KccAns` column (answer by the KCC agent)
- `querytext_language` — language of the `QueryText` column (farmer's question)

```
INPUT:  KCC_data_gov_in_0 / 01 / BIHAR / PATNA.csv
        ┌──────────┬───────────────────────┬────────────────────────────┐
        │ StateName│ KccAns                │ QueryText                  │
        ├──────────┼───────────────────────┼────────────────────────────┤
        │ BIHAR    │ धान में तना छेदक...   │ How to control stem borer? │
        └──────────┴───────────────────────┴────────────────────────────┘

OUTPUT: kcc_lid_output / 01 / BIHAR / PATNA.csv
        ┌──────────┬───────────────────────┬────────────────────────────┬─────────────────┬──────────────────────┐
        │ StateName│ KccAns                │ QueryText                  │ kccans_language │ querytext_language   │
        ├──────────┼───────────────────────┼────────────────────────────┼─────────────────┼──────────────────────┤
        │ BIHAR    │ धान में तना छेदक...   │ How to control stem borer? │ Hindi           │ English              │
        └──────────┴───────────────────────┴────────────────────────────┴─────────────────┴──────────────────────┘
```

Folder structure expected:
```
KCC_data_gov_in_0/
├── 01/  ← month
│   ├── BIHAR/  ← state
│   │   ├── PATNA.csv   ← district
│   │   └── GAYA.csv
│   └── UTTARAKHAND/
│       └── DEHRADUN.csv
├── 02/
└── ... (up to 12)
```

| Cell | What it does |
|------|-------------|
| 1 | Install packages |
| 2 | Clone IndicLID |
| 3–6 | Download + unzip models |
| 7 | cd to Inference folder |
| 8 | **Load IndicLID model** |
| 9 | Mount Drive + imports |
| 10 | **Config ← only cell you edit** |
| 11 | Detection engine |
| 12 | Debug check |
| 13 | **Main loop** — labels every row, saves output CSVs |
| 14 | **Summary report** — Month/State/District/Language counts |
| 15 | Download ZIP |

In [ ]:
%%capture
!pip install fasttext transformers langdetect pandas matplotlib tqdm openpyxl
print('Packages installed.')

In [ ]:
import os
if not os.path.isdir('/content/IndicLID'):
    !git clone https://github.com/AI4Bharat/IndicLID.git /content/IndicLID
    print('Cloned IndicLID.')
else:
    print('IndicLID already cloned.')

Cloning into '/content/IndicLID'...
remote: Enumerating objects: 345, done.
remote: Counting objects: 100% (345/345), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 345 (delta 151), reused 291 (delta 118), pack-reused 0 (from 0)
Receiving objects: 100% (345/345), 202.28 KiB | 1.20 MiB/s, done.
Resolving deltas: 100% (151/151), done.
Cloned IndicLID.


In [ ]:
%cd /content/IndicLID/Inference
!mkdir -p models
print('Working dir:', os.getcwd())

/content/IndicLID/Inference
Working dir: /content/IndicLID/Inference


In [ ]:
%cd /content/IndicLID/Inference/models
import os
if not os.path.exists('indiclid-ftn.zip'):
    !wget -q https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/indiclid-ftn.zip
    print('Downloaded ftn')
else:
    print('ftn zip already present')
if not os.path.exists('indiclid-ftr.zip'):
    !wget -q https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/indiclid-ftr.zip
    print('Downloaded ftr')
else:
    print('ftr zip already present')
if not os.path.exists('indiclid-bert.zip'):
    !wget -q https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/indiclid-bert.zip
    print('Downloaded bert')
else:
    print('bert zip already present')
print('All zips present.')

/content/IndicLID/Inference/models
Downloaded ftn
Downloaded ftr
Downloaded bert
All zips present.


In [ ]:
import zipfile, os

for zip_name, folder_name in [
    ('indiclid-ftn.zip',  'indiclid-ftn'),
    ('indiclid-ftr.zip',  'indiclid-ftr'),
    ('indiclid-bert.zip', 'indiclid-bert'),
]:
    if not os.path.isdir(folder_name):
        print(f'Unzipping {zip_name} ...')
        with zipfile.ZipFile(zip_name, 'r') as zf:
            zf.extractall('.')
        print(f'  Done -> {folder_name}/')
    else:
        print(f'Already unzipped: {folder_name}/')
print('All models ready.')

Unzipping indiclid-ftn.zip ...
  Done -> indiclid-ftn/
Unzipping indiclid-ftr.zip ...
  Done -> indiclid-ftr/
Unzipping indiclid-bert.zip ...
  Done -> indiclid-bert/
All models ready.


In [ ]:
%cd /content/IndicLID/Inference
print('Now in:', os.getcwd())

/content/IndicLID/Inference
Now in: /content/IndicLID/Inference


In [ ]:
import sys
inference_dir = '/content/IndicLID/Inference'
if inference_dir not in sys.path:
    sys.path.insert(0, inference_dir)

try:
    from ai4bharat.IndicLID import IndicLID
    indiclid_model     = IndicLID(input_threshold=0.5, roman_lid_threshold=0.6)
    INDICLID_AVAILABLE = True
    print('IndicLID loaded successfully!')
except Exception as e:
    INDICLID_AVAILABLE = False
    indiclid_model     = None
    print(f'WARNING: IndicLID failed to load: {e}')
    print('Will use Unicode heuristics for Devanagari text.')

/content/IndicLID/Inference/ai4bharat/IndicLID.py:190: SyntaxWarning: invalid escape sequence '\|'
  special_char_pattern = re.compile('[@_!#$%^&*()<>?/\|}{~:]')
/content/IndicLID/Inference/ai4bharat/IndicLID.py:194: SyntaxWarning: invalid escape sequence '\s'
  spaces = len(re.findall('\s', input))
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.75M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

IndicLID loaded successfully!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, re, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm.notebook import tqdm
from langdetect import detect, LangDetectException, DetectorFactory
DetectorFactory.seed = 42

print('All imports done!')
print(f'INDICLID_AVAILABLE = {INDICLID_AVAILABLE}')

Mounted at /content/drive
All imports done!
INDICLID_AVAILABLE = True


In [ ]:
# ══════════════════════════════════════════════════════════════════
# ONLY EDIT THIS CELL
# ══════════════════════════════════════════════════════════════════

# Root of your KCC data — contains subfolders 01/ 02/ ... 12/
INPUT_ROOT  = '/content/drive/MyDrive/kcc_pest_sample_2024_v2'

# Where output CSVs will be saved (same subfolder structure)
OUTPUT_ROOT = '/content/drive/MyDrive/kcc_lid_output'

# All 12 months — zero-padded to match your folder names
MONTHS = ['01','02','03','04','05','06','07','08','09','10','11','12']

# The two columns to detect language for
KCCANS_COL    = 'KccAns'      # agent's answer  → output col: kccans_language
QUERYTEXT_COL = 'QueryText'   # farmer's query  → output col: querytext_language

# ══════════════════════════════════════════════════════════════════
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(f'INPUT_ROOT    : {INPUT_ROOT}')
print(f'OUTPUT_ROOT   : {OUTPUT_ROOT}')
print(f'MONTHS        : {MONTHS}')
print(f'KccAns col    : {KCCANS_COL}   → output col: kccans_language')
print(f'QueryText col : {QUERYTEXT_COL} → output col: querytext_language')

INPUT_ROOT    : /content/drive/MyDrive/kcc_pest_sample_2024_v2
OUTPUT_ROOT   : /content/drive/MyDrive/kcc_lid_output
MONTHS        : ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']
KccAns col    : KccAns   → output col: kccans_language
QueryText col : QueryText → output col: querytext_language


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# LANGUAGE DETECTION ENGINE  — do not edit
# ══════════════════════════════════════════════════════════════════════════

SCRIPT_RANGES = [
    (0x0900, 0x097F, 'Devanagari'),
    (0x0980, 0x09FF, 'Bengali/Assamese'),
    (0x0A00, 0x0A7F, 'Punjabi'),
    (0x0A80, 0x0AFF, 'Gujarati'),
    (0x0B00, 0x0B7F, 'Odia'),
    (0x0B80, 0x0BFF, 'Tamil'),
    (0x0C00, 0x0C7F, 'Telugu'),
    (0x0C80, 0x0CFF, 'Kannada'),
    (0x0D00, 0x0D7F, 'Malayalam'),
    (0x1C50, 0x1C7F, 'Santali'),
    (0xABC0, 0xABFF, 'Meitei'),
]
ASSAMESE_UNIQUE = {'ৰ', 'ৱ'}

INDICLID_CODE_MAP = {
    'hin_Deva':'Hindi',       'mai_Deva':'Maithili',    'mar_Deva':'Marathi',
    'nep_Deva':'Nepali',      'kok_Deva':'Konkani',     'san_Deva':'Sanskrit',
    'doi_Deva':'Dogri',       'brx_Deva':'Bodo',        'kas_Deva':'Kashmiri',
    'ben_Beng':'Bengali',     'asm_Beng':'Assamese',    'mni_Beng':'Meitei',
    'guj_Gujr':'Gujarati',    'pan_Guru':'Punjabi',     'tam_Tamil':'Tamil',
    'tel_Telu':'Telugu',      'kan_Knda':'Kannada',     'mal_Mlym':'Malayalam',
    'ori_Orya':'Odia',        'sat_Olch':'Santali',     'mni_Meti':'Meitei',
    'urd_Arab':'Urdu',        'kas_Arab':'Kashmiri',    'snd_Arab':'Sindhi',
    'eng_Latn':'English',     'hin_Latn':'Hindi (Roman)',
    'ben_Latn':'Bengali (Roman)',    'asm_Latn':'Assamese (Roman)',
    'mai_Latn':'Maithili (Roman)',   'mar_Latn':'Marathi (Roman)',
    'guj_Latn':'Gujarati (Roman)',   'pan_Latn':'Punjabi (Roman)',
    'tam_Latn':'Tamil (Roman)',      'tel_Latn':'Telugu (Roman)',
    'kan_Latn':'Kannada (Roman)',    'mal_Latn':'Malayalam (Roman)',
    'ori_Latn':'Odia (Roman)',       'nep_Latn':'Nepali (Roman)',
    'kok_Latn':'Konkani (Roman)',    'san_Latn':'Sanskrit (Roman)',
    'doi_Latn':'Dogri (Roman)',      'brx_Latn':'Bodo (Roman)',
    'kas_Latn':'Kashmiri (Roman)',   'snd_Latn':'Sindhi (Roman)',
    'mni_Latn':'Meitei (Roman)',     'urd_Latn':'Urdu (Roman)',
    'other':   'Other',
}
LANGDETECT_MAP = {
    'en':'English','hi':'Hindi','mr':'Marathi','bn':'Bengali',
    'gu':'Gujarati','pa':'Punjabi','ta':'Tamil','te':'Telugu',
    'kn':'Kannada','ml':'Malayalam','or':'Odia','as':'Assamese',
    'ne':'Nepali','ur':'Urdu','sd':'Sindhi',
}
BHOJPURI_MARKERS = {
    'बा', 'बानि', 'रउआ',
    'काहे', 'रहल',
    'आवत', 'केही',
}

DEVANAGARI_MIN_CHARS = 5
INDICLID_MIN_SCORE   = 0.50
_indiclid_errors     = 0


def _run_indiclid(text: str) -> str:
    global _indiclid_errors
    try:
        # IMPORTANT: predict() requires a LIST — passing a plain string returns None
        # IndicLID cannot handle newlines — replace with space before passing
        clean = text.replace('\n', ' ').replace('\r', ' ').strip()
        result = indiclid_model.predict(clean)
        if not result:
            return 'Hindi'
        top      = result[0]       # ('hin_Deva', 0.98)
        raw_code = str(top[0])
        score    = float(top[1])
        if score < INDICLID_MIN_SCORE:
            return 'Hindi'
        return INDICLID_CODE_MAP.get(raw_code, f'Devanagari ({raw_code})')
    except Exception as e:
        _indiclid_errors += 1
        if _indiclid_errors <= 5:
            print(f'[IndicLID ERROR #{_indiclid_errors}] {repr(text[:40])}: {e}')
        return 'Hindi'


def detect_language(text: str) -> str:
    if not isinstance(text, str):
        return 'Unknown'
    text = text.strip()
    if len(text) < 3:
        return 'Unknown'
    total_chars = sum(1 for c in text if not c.isspace())
    if total_chars == 0:
        return 'Unknown'

    block_counts = {}
    for start, end, label in SCRIPT_RANGES:
        cnt = sum(1 for c in text if start <= ord(c) <= end)
        if cnt > 0:
            block_counts[label] = block_counts.get(label, 0) + cnt

    if block_counts:
        dominant_label = max(block_counts, key=block_counts.get)
        dominant_count = block_counts[dominant_label]
        if dominant_count / total_chars >= 0.20:
            if dominant_label == 'Bengali/Assamese':
                return 'Assamese' if set(text) & ASSAMESE_UNIQUE else 'Bengali'
            if dominant_label == 'Devanagari':
                if INDICLID_AVAILABLE and dominant_count >= DEVANAGARI_MIN_CHARS:
                    label = _run_indiclid(text)
                    if label == 'Hindi' and set(text.split()) & BHOJPURI_MARKERS:
                        return 'Bhojpuri'
                    return label
                return 'Marathi' if 'ळ' in text else 'Hindi/Devanagari'
            return dominant_label

    try:
        code = detect(text)
        return LANGDETECT_MAP.get(code, f'Other ({code})')
    except LangDetectException:
        return 'Unknown'


# Quick sanity tests
print('Sanity tests:')
for txt, exp in [
    ('My crop has pest infestation',                       'English'),
    ('How to control stem borer in paddy?',                'English'),
    ('నా పంటలో చీడ', 'Telugu'),
    ('என் பயிரில்',   'Tamil'),
    ('ਮੇਰੀ ਫਸਲ ਤੇ ਕੀੜੇ', 'Punjabi'),
    ('ਮારા પાકમાં', 'Gujarati'),
]:
    got = detect_language(txt)
    print(f'  {"OK" if got==exp else "FAIL"}  Expected:{exp:<16} Got:{got}')
print('Devanagari (shows real IndicLID output):')
for txt in [
    'मेरी फसल में कीड़े लग गए हैं',
    'माझ्या शेतात किडे आले आहेत',
    'हमार खेत में किरा लागल बा',
    'अछि हमर खेत मे समस्या',
]:
    print(f'  {txt[:48]!r}  ->  {detect_language(txt)}')
print(f'IndicLID errors: {_indiclid_errors}')
print('Detection engine ready!')


Sanity tests:
  OK  Expected:English          Got:English
  OK  Expected:English          Got:English
  OK  Expected:Telugu           Got:Telugu
  OK  Expected:Tamil            Got:Tamil
  OK  Expected:Punjabi          Got:Punjabi
  OK  Expected:Gujarati         Got:Gujarati
Devanagari (shows real IndicLID output):
  'मेरी फसल में कीड़े लग गए हैं'  ->  Hindi
  'माझ्या शेतात किडे आले आहेत'  ->  Hindi
  'हमार खेत में किरा लागल बा'  ->  Bhojpuri
  'अछि हमर खेत मे समस्या'  ->  Hindi
IndicLID errors: 0
Detection engine ready!


In [ ]:
# ── DEBUG — run this before Cell 12 to verify IndicLID is working ────────
print('='*60)
print(f'INDICLID_AVAILABLE: {INDICLID_AVAILABLE}')
if INDICLID_AVAILABLE:
    test = 'मेरी फसल में कीड़े'
    # IMPORTANT: predict() needs a LIST — plain string returns None
    raw  = indiclid_model.predict(test)
    print(f'Raw predict() output : {raw}')
    if raw and len(raw) > 0:
        print(f'Code                 : {raw[0][0]}')
        print(f'Score                : {raw[0][1]}')
        print(f'Mapped to            : {INDICLID_CODE_MAP.get(str(raw[0][0]), "NOT IN MAP")}')
        print('\nIndicLID working correctly. Run Cell 12.')
    else:
        print('predict() returned empty/None. Check model loading in Cell 7.')
else:
    print('IndicLID not loaded. Re-run Cell 7.')
print('='*60)


INDICLID_AVAILABLE: True
Raw predict() output : None
predict() returned empty/None. Check model loading in Cell 7.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# MAIN PROCESSING LOOP
#
# Walks: INPUT_ROOT / month / state / district.csv
# Adds two columns to every row:
#   kccans_language    = language of KccAns
#   querytext_language = language of QueryText
# Saves output to OUTPUT_ROOT with same folder path.
# ══════════════════════════════════════════════════════════════════════════

_indiclid_errors = 0   # reset before full run
processed_rows   = []  # one entry per (month, state, district, language, column)
skipped_files    = []  # files we could not process

for month in MONTHS:
    month_dir = os.path.join(INPUT_ROOT, month)
    if not os.path.isdir(month_dir):
        print(f'[SKIP] Month folder not found: {month_dir}')
        continue

    state_folders = sorted([
        d for d in os.listdir(month_dir)
        if os.path.isdir(os.path.join(month_dir, d))
    ])
    print(f'\n═══ MONTH {month} — {len(state_folders)} states ═══')

    for state_folder in tqdm(state_folders, desc=f'Month {month}'):
        state_dir = os.path.join(month_dir, state_folder)

        csv_files = sorted(
            glob.glob(os.path.join(state_dir, '*.csv'))  +
            glob.glob(os.path.join(state_dir, '*.CSV'))  +
            glob.glob(os.path.join(state_dir, '*.xlsx')) +
            glob.glob(os.path.join(state_dir, '*.XLSX'))
        )
        if not csv_files:
            print(f'  [SKIP] No files in {state_folder}')
            continue

        for fpath in csv_files:
            fname    = os.path.basename(fpath)
            district = os.path.splitext(fname)[0].upper()  # filename = district name

            # ── Read ─────────────────────────────────────────────────────
            try:
                if fpath.lower().endswith('.csv'):
                    try:    df = pd.read_csv(fpath, encoding='utf-8',  low_memory=False)
                    except: df = pd.read_csv(fpath, encoding='latin1', low_memory=False)
                else:
                    df = pd.read_excel(fpath)
            except Exception as e:
                print(f'  [READ ERROR] {fname}: {e}')
                skipped_files.append({'month':month,'state':state_folder,'file':fname,'reason':str(e)})
                continue

            # ── Detect KccAns language ────────────────────────────────────
            # Find column case-insensitively
            def find_col(df, name):
                if name in df.columns: return name
                for c in df.columns:
                    if c.lower().strip() == name.lower(): return c
                return None

            kccans_col    = find_col(df, KCCANS_COL)
            querytext_col = find_col(df, QUERYTEXT_COL)

            if kccans_col is None and querytext_col is None:
                print(f'  [WARN] Neither {KCCANS_COL} nor {QUERYTEXT_COL} found in {fname}')
                print(f'         Columns: {list(df.columns)[:8]}')
                skipped_files.append({'month':month,'state':state_folder,'file':fname,'reason':'columns not found'})
                continue

            if kccans_col:
                df['kccans_language'] = df[kccans_col].apply(detect_language)
            else:
                df['kccans_language'] = 'Unknown'   # column missing in this file

            if querytext_col:
                df['querytext_language'] = df[querytext_col].apply(detect_language)
            else:
                df['querytext_language'] = 'Unknown'

            # ── Save output CSV ───────────────────────────────────────────
            out_dir  = os.path.join(OUTPUT_ROOT, month, state_folder)
            os.makedirs(out_dir, exist_ok=True)
            out_path = os.path.join(out_dir, fname)
            df.to_csv(out_path, index=False, encoding='utf-8-sig')

            # ── Record for summary ────────────────────────────────────────
            state_upper = state_folder.upper()
            for lang, cnt in df['kccans_language'].value_counts().items():
                processed_rows.append({
                    'month': month, 'state': state_upper, 'district': district,
                    'column': 'KccAns', 'language': lang, 'count': cnt
                })
            for lang, cnt in df['querytext_language'].value_counts().items():
                processed_rows.append({
                    'month': month, 'state': state_upper, 'district': district,
                    'column': 'QueryText', 'language': lang, 'count': cnt
                })

print(f'\n\nAll files processed!')
print(f'IndicLID errors : {_indiclid_errors}')
print(f'Skipped files   : {len(skipped_files)}')
print(f'Output saved to : {OUTPUT_ROOT}')
if skipped_files:
    print('Skipped file details:')
    for s in skipped_files:
        print(f"  {s['month']}/{s['state']}/{s['file']} — {s['reason']}")


═══ MONTH 01 — 27 states ═══


Month 01:   0%|          | 0/27 [00:00<?, ?it/s]

[IndicLID ERROR #1] '-कृषि यंत्रों की सब्सिडी की जानकारी के ल': 'token_type_ids'
[IndicLID ERROR #2] 'सब्सिडीज की जानकारी के लिए लॉगिन करें  h': 'token_type_ids'
[IndicLID ERROR #3] 'सब्सिडीज की जानकारी के लिए लॉगिन करें  h': 'token_type_ids'
[IndicLID ERROR #4] 'किसान भाई करोंदा के बारे में जानकारी के ': 'token_type_ids'
[IndicLID ERROR #5] 'सेब का स्प्रे शेड्यूल लिंक\nhttps://eudya': 'token_type_ids'

═══ MONTH 02 — 28 states ═══


Month 02:   0%|          | 0/28 [00:00<?, ?it/s]


═══ MONTH 03 — 27 states ═══


...


═══ MONTH 11 — 28 states ═══


Month 11:   0%|          | 0/28 [00:00<?, ?it/s]


═══ MONTH 12 — 28 states ═══


Month 12:   0%|          | 0/28 [00:00<?, ?it/s]



All files processed!
IndicLID errors : 148
Skipped files   : 0
Output saved to : /content/drive/MyDrive/kcc_lid_output


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# SUMMARY REPORT
#
# Prints:  Month → State → District → Language counts  (for both columns)
# Saves :  summary_month_state_district_language.csv
# ══════════════════════════════════════════════════════════════════════════

if not processed_rows:
    print('No data. Run Cell 12 first.')
else:
    summary_df = pd.DataFrame(processed_rows)
    summary_df = summary_df.sort_values(
        ['month','state','district','column','count'],
        ascending=[True,True,True,True,False]
    ).reset_index(drop=True)

    # Save full long-format summary
    summary_csv = os.path.join(OUTPUT_ROOT, 'summary_month_state_district_language.csv')
    summary_df.to_csv(summary_csv, index=False, encoding='utf-8-sig')
    print(f'Summary CSV saved: {summary_csv}')
    print(f'Total records    : {len(summary_df):,}')

    # ── Printed report ────────────────────────────────────────────────────
    print('\n' + '='*75)
    print('LANGUAGE DISTRIBUTION REPORT')
    print('Columns: month | state | district | column | language | count')
    print('='*75)

    cur_month = None
    cur_state = None
    cur_dist  = None

    for _, row in summary_df.iterrows():
        if row['month'] != cur_month:
            cur_month = row['month']
            cur_state = None
            print(f'\n┌── MONTH {cur_month} ' + '─'*55)

        if row['state'] != cur_state:
            cur_state = row['state']
            cur_dist  = None
            state_total = summary_df[
                (summary_df['month']==cur_month) &
                (summary_df['state']==cur_state)
            ]['count'].sum()
            print(f'│')
            print(f'│  ▶ {cur_state:<25}  total rows: {state_total:,}')

        if row['district'] != cur_dist:
            cur_dist = row['district']
            print(f'│     ── {cur_dist}')

        col_label = 'KccAns   ' if row['column']=='KccAns' else 'QueryText'
        print(f'│         {col_label}  {row["language"]:<25} {row["count"]:>6,} rows')

    print('│')
    print('└' + '─'*74)

    # ── Overall totals split by column ───────────────────────────────────
    print('\n' + '='*75)
    print('OVERALL TOTALS')
    print('='*75)
    for col_name in ['KccAns', 'QueryText']:
        sub = summary_df[summary_df['column']==col_name]
        tot = sub.groupby('language')['count'].sum().sort_values(ascending=False).reset_index()
        tot.columns = ['Language', 'Total Rows']
        tot['%'] = (tot['Total Rows'] / tot['Total Rows'].sum() * 100).round(1)
        print(f'\n  {col_name} (grand total rows: {tot["Total Rows"].sum():,})')
        print('  ' + '-'*45)
        for _, r in tot.iterrows():
            bar = '█' * int(r['%'] / 2)
            print(f'  {r["Language"]:<25} {r["Total Rows"]:>8,}  ({r["%"]:>5.1f}%)  {bar}')

    display(summary_df.head(30))

Streaming output truncated to the last 5000 lines.
│         KccAns     Hindi                         50 rows
│         QueryText  English                       50 rows
│     ── BALAGHAT
│         KccAns     Hindi                         24 rows
│         QueryText  English                       24 rows
│     ── BARWANI
│         KccAns     Hindi                          8 rows
│         QueryText  English                        8 rows
│     ── BETUL
│         KccAns     Hindi                         50 rows
│         QueryText  English                       50 rows
│     ── BHIND
│         KccAns     Hindi                         50 rows
│         QueryText  English                       50 rows
│     ── BHOPAL
│         KccAns     Hindi                         50 rows
│         QueryText  English                       50 rows
│     ── BURHANPUR
│         KccAns     Hindi                          8 rows
│         QueryText  English                        8 rows
│     ── CHHATARPUR
│  

,month,state,district,column,language,count
0,01,ANDHRA PRADESH,0,KccAns,Telugu,1
1,01,ANDHRA PRADESH,0,QueryText,English,1
2,01,ANDHRA PRADESH,ANANTPUR,KccAns,Telugu,36
3,01,ANDHRA PRADESH,ANANTPUR,KccAns,English,12
4,01,ANDHRA PRADESH,ANANTPUR,KccAns,Other (de),1
5,01,ANDHRA PRADESH,ANANTPUR,KccAns,Other (id),1
6,01,ANDHRA PRADESH,ANANTPUR,QueryText,English,28
7,01,ANDHRA PRADESH,ANANTPUR,QueryText,Other (so),9
8,01,ANDHRA PRADESH,ANANTPUR,QueryText,Other (de),6
9,01,ANDHRA PRADESH,ANANTPUR,QueryText,Other (fr),4


In [ ]:
import shutil
from google.colab import files
zip_base = '/content/kcc_lid_output_export'
shutil.make_archive(zip_base, 'zip', OUTPUT_ROOT)
files.download(zip_base + '.zip')
print('Download started!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started!
